# Reproducing the ODIL solution

For a bit of context, this notebook aims to reproduce the ODIL solution in figure 1 of [the original paper](https://arxiv.org/pdf/2205.04611). 

Figure 1 solves the 1D wave equation,

$$\frac 1 {c^2} u_{tt} = u_{xx},$$

with $c = 1$.

I will attempt to implement ODIL to solve the 1D wave equation in this notebook. The general order of proceedings will be approximately as follows

1. Discretise
2. Formulate the functional (discrete PDE residual + ICs + PML BCs)
3. Optimise (L-BFGS-B, Newton)

## Discretisation

The paper applies ODIL over a rectangular domain $x \in (-1, 1),\, t \in (0, 1)$ on a $25 \times 25$ grid. This then leads to $\Delta x = 0.08,\, \Delta t = 0.04$.

We can approximate the wave equation by finite difference stencils as they do, too. They use central differences derived from Taylor series expansions:

$$u_{tt}(t, x) = \frac {u^{n+1}_i -2u^n_i + u^{n-1}_i} {\Delta t^2} + \mathcal{O}(\Delta t^2), \tag{temporal derivative}$$
$$u_{xx}(t, x)= \frac {u^n_{i+1} -2u^n_i + u^{n}_{i+1}} {\Delta x^2} + \mathcal{O}(\Delta x^2). \tag{spatial derivative}$$

Thus in explicit form we seek to solve

$$\frac {u^{n+1}_i -2u^n_i + u^{n-1}_i} {\Delta t^2} = \frac {u^n_{i+1} -2u^n_i + u^{n}_{i+1}} {\Delta x^2},$$

since $c = 1$.

## Formulating the functional

In ODIL, the PDE is enforced on the grid implicitly via the functional. It consists of:

- A PDE residual term
- An initial condition(s) term
- A boundary conditon(s) term

The first is simple. Since $c = 1$ and we do not have a source, the PDE residual is

$$\mathcal{R}_\text{PDE} = u_{tt} - u_{xx},$$

which is neccesarily 0 if the PDE is satsified. In discrete form, we can write

$$\mathcal{R}_\text{PDE} = \frac {u^{n+1}_i -2u^n_i + u^{n-1}_i} {\Delta t^2} - \frac {u^n_{i+1} -2u^n_i + u^{n}_{i+1}} {\Delta x^2}$$